<a href="https://colab.research.google.com/github/PYMaksim/Corporate-RAG-Assistant/blob/main/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ENG: Installing the vector database library. # РУС: Устанавливаем библиотеку для векторной базы данных.
!pip install --upgrade --quiet sentence-transformers chromadb tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7

In [3]:
# --- BLOCK 1: SETUP ---

# ENG: Installing necessary libraries.
# 1. sentence-transformers: The tool (Encoder) that converts human text into mathematical vectors (Embeddings).
# 2. chromadb: A simple vector database where we will store those vectors.
# 3. tqdm: A visual progress bar to see how long the encoding process takes.
# РУС: Устанавливаем необходимые библиотеки.
# 1. sentence-transformers: Инструмент (Энкодер), превращающий человеческий текст в математические векторы (Эмбеддинги).
# 2. chromadb: Векторная база данных, где мы будем хранить эти векторы.
# 3. tqdm: Красивая полоса загрузки, чтобы видеть прогресс обработки.
# !pip install sentence-transformers chromadb tqdm

# --- IMPORTS ---

# ENG: Importing the Encoder model. It "understands" the meaning of text.
# РУС: Импортируем модель-энкодер. Она "понимает" смысл текста.
from sentence_transformers import SentenceTransformer

# ENG: Importing the Vector Database client.
# РУС: Импортируем клиент для работы с векторной базой данных.
import chromadb

# ENG: Importing the progress bar for long operations.
# РУС: Импортируем полосу прогресса для долгих операций.
from tqdm import tqdm

# ENG: Importing text wrapper for splitting long documents into chunks.
# РУС: Импортируем инструмент для нарезки текста на куски.
import textwrap

# --- INITIALIZATION ---

# ENG: Connecting to the database.
# If the "chroma_db" folder doesn't exist, it will be created automatically.
# РУС: Подключаемся к базе данных.
# Если папки "chroma_db" нет, она создастся автоматически.
client = chromadb.PersistentClient(path="chroma_db")

# ENG: Creating or getting a "collection" (like a table in SQL).
# This is where our instruction vectors will live.
# РУС: Создаем или получаем "коллекцию" (аналог таблицы в SQL).
# Здесь будут жить векторы наших инструкций.
collection = client.get_or_create_collection(name="company_instructions")

In [4]:
# --- BLOCK 2: CHUNKING & DATA PREP ---

# ENG: Our knowledge base (15 sentences, bilingual).
# РУС: Наша база знаний (15 предложений, двуязычные).
raw_docs = [
    # --- Block 1: Vacation & HR ---
    "ENG: To apply for vacation, an employee must submit a written request 14 days in advance. The HR department reviews the balance of days. If the balance is positive, the order is signed within 3 days. RUS: Для оформления отпуска сотрудник пишет заявление за 14 дней. Отдел кадров проверяет остаток дней. Если дни есть, приказ подписывают за 3 дня.",

    # --- Block 2: Bonuses ---
    "ENG: Bonuses are paid quarterly based on KPI achievement. The minimum threshold is 100% of the plan. If you exceed the plan by 20%, you get a multiplier. RUS: Премии выплачиваются раз в квартал за выполнение KPI. Минимальный порог — 100% плана. Перевыполнил на 20% — включается коэффициент.",

    # --- Block 3: IT Support ---
    "ENG: If your computer breaks, create a ticket in the IT portal. Describe the error and attach a screenshot. IT response time is 4 hours. RUS: Если сломался компьютер, создай заявку в портале IT. Опиши ошибку и приложи скриншот. Время реакции сисадмина — 4 часа.",

    # --- Block 4: Security ---
    "ENG: Passwords must be changed every 90 days. Use at least 12 characters including numbers. Do not share your credentials. RUS: Пароли меняются раз в 90 дней. Минимум 12 знаков, включая цифры. Не передавайте ключи коллегам.",

    # --- Block 5: Schedule ---
    "ENG: Working hours are from 9:00 to 18:00. Lunch break is 1 hour from 13:00 to 14:00. Remote work is allowed on Fridays. RUS: График с 9:00 до 18:00. Обед 1 час (13:00-14:00). В пятницу можно работать из дома."
]

# ENG: Load the model that understands both languages.
# РУС: Загружаем модель, понимающую оба языка.
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# ENG: Function to split long text into 500-char chunks.
# РУС: Функция нарезки длинного текста на куски по 500 символов.
def chunk_text(text, chunk_size=500):
    return textwrap.wrap(text, width=chunk_size)

documents = []
ids = []

# ENG: Process each document: split into chunks and create IDs.
# РУС: Обрабатываем каждый документ: режем на куски и создаем ID.
for i, doc in enumerate(raw_docs):
    chunks = chunk_text(doc)
    for chunk in chunks:
        documents.append(chunk)
        ids.append(f"doc_{i}_{len(ids)}") # Unique ID for each chunk

print(f"Total chunks created: {len(documents)}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Total chunks created: 5


Warning: This is not an error. Hugging Face informs us that we are downloading the model without an API key (as a guest). Download speed might be slightly lower, but it does not affect the result.
Предупреждение (Warning): Это не ошибка. Hugging Face просто сообщает, что мы скачиваем модель без авторизации (как гость). Скорость скачивания может быть чуть ниже, но на результат это не влияет.
__________________________________________________
Downloading Model: We are downloading the neural network weights (*.safetensors and config.json). Total size is ~500 MB. The line Loading weights: 100% means the model is fully loaded into RAM and ready to process data.
Загрузка модели: Мы скачиваем веса нейросети (файлы *.safetensors и config.json). Общий вес около 500 МБ. Строка Loading weights: 100% означает, что модель полностью загружена в оперативную память Colab и готова к работе.
____________________________________________________
Data Prep Result: We had 5 large text blocks (instructions). The chunking function split them into 5 fragments (chunks). Now we will convert each fragment into a separate mathematical vector.
Результат подготовки данных: Было 5 больших текстовых блоков (инструкций). Функция нарезки (chunking) разделила их на 5 фрагментов (чанков). Теперь мы будем превращать каждый фрагмент в отдельный математический вектор.



In [5]:
# --- BLOCK 3: EMBEDDING ---

# ENG: Transforming text chunks into mathematical vectors.
# РУС: Превращаем текстовые куски в математические векторы.
print("Vectorizing documents...")
embeddings = model.encode(documents, show_progress_bar=True)

Vectorizing documents...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [6]:
# --- BLOCK 4: UPLOAD TO DB ---

# ENG: Adding vectors and metadata to ChromaDB.
# РУС: Добавляем векторы и тексты в нашу базу.
collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings
)

print("Knowledge base is ready. Vectors are stored.")

Knowledge base is ready. Vectors are stored.


In [7]:
# --- BLOCK 5: RETRIEVAL & TESTING ---

def search_knowledge(query_text, top_k=3):
    # ENG: Convert the user's question into a vector.
    # РУС: Превращаем вопрос в вектор.
    query_vector = model.encode([query_text])

    # ENG: Search for the most similar vectors in the DB.
    # РУС: Ищем самые похожие векторы в базе.
    results = collection.query(
        query_embeddings=query_vector,
        n_results=top_k
    )

    # ENG: Print the found documents.
    # РУС: Выводим найденные инструкции.
    for i, doc in enumerate(results['documents']):
        print(f"--- MATCH {i+1} ---")
        print(doc)
    print("-" * 20)

# --- TEST CASES (RU) ---
print(">>> TEST 1 (RU): Question EXISTS in base")
search_knowledge("Как оформить отпуск и сколько дней нужно отработать?")

print("\n>>> TEST 2 (RU): Question NOT in base (Hard query)")
search_knowledge("Как мне уволиться по собственному желанию и получить расчет?")

# --- TEST CASES (EN) ---
print("\n>>> TEST 3 (EN): Question EXISTS in base")
search_knowledge("What are the rules for changing my password?")

print("\n>>> TEST 4 (EN): Question NOT in base")
search_knowledge("How do I order a taxi for a business trip?")

>>> TEST 1 (RU): Question EXISTS in base
--- MATCH 1 ---
['ENG: To apply for vacation, an employee must submit a written request 14 days in advance. The HR department reviews the balance of days. If the balance is positive, the order is signed within 3 days. RUS: Для оформления отпуска сотрудник пишет заявление за 14 дней. Отдел кадров проверяет остаток дней. Если дни есть, приказ подписывают за 3 дня.', 'ENG: Working hours are from 9:00 to 18:00. Lunch break is 1 hour from 13:00 to 14:00. Remote work is allowed on Fridays. RUS: График с 9:00 до 18:00. Обед 1 час (13:00-14:00). В пятницу можно работать из дома.', 'ENG: Passwords must be changed every 90 days. Use at least 12 characters including numbers. Do not share your credentials. RUS: Пароли меняются раз в 90 дней. Минимум 12 знаков, включая цифры. Не передавайте ключи коллегам.']
--------------------

>>> TEST 2 (RU): Question NOT in base (Hard query)
--- MATCH 1 ---
['ENG: If your computer breaks, create a ticket in the IT porta